In [9]:
import glob
import pandas as pd 

files = glob.glob("/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/pv_extracted_csv/*.[cC][sS][vV]")

print("Files found:", len(files))
print(files[:5])

Files found: 1337
['/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/pv_extracted_csv/PUBLIC_ROOFTOP_PV_ACTUAL_SATELLITE_20260303080000_0000000506040725.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/pv_extracted_csv/PUBLIC_ROOFTOP_PV_ACTUAL_SATELLITE_20260318080000_0000000508413330.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/pv_extracted_csv/PUBLIC_ROOFTOP_PV_ACTUAL_SATELLITE_20260318170000_0000000508478526.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/pv_extracted_csv/PUBLIC_ROOFTOP_PV_ACTUAL_SATELLITE_20260321220000_0000000509051667.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/pv_extracted_csv/PUBLIC_ROOFTOP_PV_ACTUAL_SATELLITE_20260320060000_0000000508757363.csv']


In [10]:
import pandas as pd
import glob

files = sorted(
    glob.glob("/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/pv_extracted_csv/*.csv")
)

print("Files found:", len(files))

all_rows = []

for f in files:
    with open(f, "r", errors="ignore") as file:
        for line in file:
            row = line.strip().split(",")
            all_rows.append(row)

print("Total rows loaded:", len(all_rows))

Files found: 1337
Total rows loaded: 10696


In [11]:
pv_raw = pd.DataFrame(all_rows)

print("Raw shape:", pv_raw.shape)
pv_raw.head(10)

Raw shape: (10696, 10)


,0,1,2,3,4,5,6,7,8,9
0,C,NEMP.WORLD,ROOFTOP_PV_ACTUAL_SATELLITE,AEMO,PUBLIC,2026/02/26,00:00:18,0000000505153089,DEMAND,0000000505153087
1,I,ROOFTOP,ACTUAL,2,INTERVAL_DATETIME,REGIONID,POWER,QI,TYPE,LASTCHANGED
2,D,ROOFTOP,ACTUAL,2,"""2026/02/25 23:30:00""",NSW1,0,0.6,SATELLITE,"""2026/02/25 23:50:18"""
3,D,ROOFTOP,ACTUAL,2,"""2026/02/25 23:30:00""",QLD1,0,0.6,SATELLITE,"""2026/02/25 23:50:18"""
4,D,ROOFTOP,ACTUAL,2,"""2026/02/25 23:30:00""",SA1,0,0.6,SATELLITE,"""2026/02/25 23:50:19"""
5,D,ROOFTOP,ACTUAL,2,"""2026/02/25 23:30:00""",TAS1,0,0.6,SATELLITE,"""2026/02/25 23:50:19"""
6,D,ROOFTOP,ACTUAL,2,"""2026/02/25 23:30:00""",VIC1,0,0.6,SATELLITE,"""2026/02/25 23:50:19"""
7,C,"""END OF REPORT""",8,None,None,None,None,None,None,None
8,C,NEMP.WORLD,ROOFTOP_PV_ACTUAL_SATELLITE,AEMO,PUBLIC,2026/02/26,00:30:19,0000000505156090,DEMAND,0000000505156088
9,I,ROOFTOP,ACTUAL,2,INTERVAL_DATETIME,REGIONID,POWER,QI,TYPE,LASTCHANGED


In [21]:
pv_raw[0] = pv_raw[0].astype(str).str.replace('"', '', regex=False).str.strip()

print(pv_raw[0].unique()[:10])

pv_df = pv_raw[pv_raw[0] == "D"].copy()
print("After D filter:", pv_df.shape)

['C' 'I' 'D']
After D filter: (6685, 10)


In [24]:
pv_raw[0] = pv_raw[0].astype(str).str.replace('"', '', regex=False).str.strip()

print(pv_raw[0].unique()[:10])

pv_df = pv_raw[pv_raw[0] == "D"].copy()
print("After D filter:", pv_df.shape)

['C' 'I' 'D']
After D filter: (6685, 10)


In [25]:
print("pv_raw shape:", pv_raw.shape)
print("pv_df shape:", pv_df.shape)
print("Columns available:", pv_df.columns.tolist())

pv_raw shape: (10696, 10)
pv_df shape: (6685, 10)
Columns available: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [26]:
# Clean row-type column first
pv_raw[0] = pv_raw[0].astype(str).str.replace('"', '', regex=False).str.strip()

# Keep only data rows
pv_df = pv_raw[pv_raw[0] == "D"].copy()
print("After D filter:", pv_df.shape)

# Check enough columns exist
print("pv_df columns:", pv_df.columns.tolist())
print(pv_df.iloc[:5, :10])

# Extract needed columns
pv_df = pv_df[[5, 6, 7]].copy()
pv_df.columns = ["SETTLEMENTDATE", "REGIONID", "PV_MW"]

# Clean strings
pv_df["SETTLEMENTDATE"] = pv_df["SETTLEMENTDATE"].astype(str).str.replace('"', '', regex=False).str.strip()
pv_df["REGIONID"] = pv_df["REGIONID"].astype(str).str.replace('"', '', regex=False).str.strip()
pv_df["PV_MW"] = pv_df["PV_MW"].astype(str).str.replace('"', '', regex=False).str.strip()

# Convert types
pv_df["SETTLEMENTDATE"] = pd.to_datetime(
    pv_df["SETTLEMENTDATE"],
    format="%Y/%m/%d %H:%M:%S",
    errors="coerce"
)
pv_df["PV_MW"] = pd.to_numeric(pv_df["PV_MW"], errors="coerce")

print(pv_df.head())
print(pv_df["REGIONID"].unique())
print(pv_df["SETTLEMENTDATE"].min(), pv_df["SETTLEMENTDATE"].max())

After D filter: (6685, 10)
pv_df columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
   0        1       2  3                      4     5  6    7          8  \
2  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"  NSW1  0  0.6  SATELLITE   
3  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"  QLD1  0  0.6  SATELLITE   
4  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"   SA1  0  0.6  SATELLITE   
5  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"  TAS1  0  0.6  SATELLITE   
6  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"  VIC1  0  0.6  SATELLITE   

                       9  
2  "2026/02/25 23:50:18"  
3  "2026/02/25 23:50:18"  
4  "2026/02/25 23:50:19"  
5  "2026/02/25 23:50:19"  
6  "2026/02/25 23:50:19"  
  SETTLEMENTDATE REGIONID  PV_MW
2            NaT        0    0.6
3            NaT        0    0.6
4            NaT        0    0.6
5            NaT        0    0.6
6            NaT        0    0.6
['0' '65.809' '61.308' ... '0.628' '24.21' '2.422']
NaT NaT


In [27]:
# Keep only D rows
pv_raw[0] = pv_raw[0].astype(str).str.replace('"', '', regex=False).str.strip()
pv_df = pv_raw[pv_raw[0] == "D"].copy()

print("After D filter:", pv_df.shape)
print(pv_df.iloc[:5, :10])

# Correct columns for this PV structure
pv_df = pv_df[[4, 5, 6]].copy()
pv_df.columns = ["SETTLEMENTDATE", "REGIONID", "PV_MW"]

# Clean strings
pv_df["SETTLEMENTDATE"] = (
    pv_df["SETTLEMENTDATE"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

pv_df["REGIONID"] = (
    pv_df["REGIONID"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

pv_df["PV_MW"] = (
    pv_df["PV_MW"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

# Convert types
pv_df["SETTLEMENTDATE"] = pd.to_datetime(
    pv_df["SETTLEMENTDATE"],
    format="%Y/%m/%d %H:%M:%S",
    errors="coerce"
)

pv_df["PV_MW"] = pd.to_numeric(pv_df["PV_MW"], errors="coerce")

# Debug before filter
print("Unique regions:", pv_df["REGIONID"].unique())
print("Min date:", pv_df["SETTLEMENTDATE"].min())
print("Max date:", pv_df["SETTLEMENTDATE"].max())
print("Null dates:", pv_df["SETTLEMENTDATE"].isna().sum())

# Filter VIC and date range
pv_df = pv_df[pv_df["REGIONID"] == "VIC1"].copy()

pv_df = pv_df[
    (pv_df["SETTLEMENTDATE"] >= "2026-03-09") &
    (pv_df["SETTLEMENTDATE"] < "2026-03-17")
].copy()

# Final clean
pv_df = pv_df.drop_duplicates(subset=["SETTLEMENTDATE", "REGIONID"])
pv_df = pv_df.sort_values("SETTLEMENTDATE").reset_index(drop=True)

print("Filtered shape:", pv_df.shape)
print(pv_df.head())
print(pv_df.tail())
print(pv_df.isna().sum())

After D filter: (6685, 10)
   0        1       2  3                      4     5  6    7          8  \
2  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"  NSW1  0  0.6  SATELLITE   
3  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"  QLD1  0  0.6  SATELLITE   
4  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"   SA1  0  0.6  SATELLITE   
5  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"  TAS1  0  0.6  SATELLITE   
6  D  ROOFTOP  ACTUAL  2  "2026/02/25 23:30:00"  VIC1  0  0.6  SATELLITE   

                       9  
2  "2026/02/25 23:50:18"  
3  "2026/02/25 23:50:18"  
4  "2026/02/25 23:50:19"  
5  "2026/02/25 23:50:19"  
6  "2026/02/25 23:50:19"  
Unique regions: ['NSW1' 'QLD1' 'SA1' 'TAS1' 'VIC1']
Min date: 2026-02-25 23:30:00
Max date: 2026-03-25 23:00:00
Null dates: 0
Filtered shape: (377, 3)
       SETTLEMENTDATE REGIONID  PV_MW
0 2026-03-09 00:00:00     VIC1    0.0
1 2026-03-09 00:30:00     VIC1    0.0
2 2026-03-09 01:00:00     VIC1    0.0
3 2026-03-09 01:30:00     VIC1    0.0
4 2026-03-09 

In [28]:
full_index = pd.date_range(
    start="2026-03-09 00:00:00",
    end="2026-03-16 23:30:00",
    freq="30min"
)

missing = full_index.difference(pv_df["SETTLEMENTDATE"])

print("Expected rows:", len(full_index))
print("Actual rows:", len(pv_df))
print("Missing rows:", len(missing))
print(missing)

Expected rows: 384
Actual rows: 377
Missing rows: 7
DatetimeIndex(['2026-03-10 16:30:00', '2026-03-10 17:00:00',
               '2026-03-10 17:30:00', '2026-03-10 18:00:00',
               '2026-03-10 18:30:00', '2026-03-10 19:00:00',
               '2026-03-10 19:30:00'],
              dtype='datetime64[ns]', freq='30min')


In [29]:
pv_df.to_csv(
    "/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/processed/pv_clean.csv",
    index=False
)

print("✅ PV saved successfully")

✅ PV saved successfully


In [30]:
pv_week = pv_df[
    (pv_df["SETTLEMENTDATE"] >= "2026-03-09") &
    (pv_df["SETTLEMENTDATE"] < "2026-03-16")
].copy()

pv_week.to_csv(
    "/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/processed/pv_week.csv",
    index=False
)

In [31]:
pv_d1 = pv_df[
    (pv_df["SETTLEMENTDATE"] >= "2026-03-16") &
    (pv_df["SETTLEMENTDATE"] < "2026-03-17")
].copy()

pv_d1.to_csv(
    "/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/processed/pv_d1.csv",
    index=False
)